# Demo 7 — Telemetry, attribution, hard caps and circuit breakers

**AI Cost Management and Token Utilization** · Module 4 · ~12 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. In twenty minutes you can stand up the measurement layer most enterprises took six months to build.
2. Attribution requires **tags on every call** — retrofitting it is far more expensive.
3. A budget you have never tested firing is not a control.
4. Agent circuit breakers stop the incident; forecasts only describe it.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
import anthropic, time, uuid, json
import pandas as pd
from collections import defaultdict
client = anthropic.Anthropic()
MODEL = 'claude-haiku-4-5'   # cheap tier — this notebook makes a lot of calls

---
## 1. The instrumented client

Emits OpenTelemetry GenAI semantic-convention attributes. Note the status caveat:
as of July 2026 **no GenAI-specific span, metric or attribute is marked Stable** — adopt it
as the direction of travel and pin your version.

Also note the **normalisation trap**: OpenAI counts cached tokens *inside* `prompt_tokens`;
Anthropic reports `cache_read_input_tokens` in a separate bucket. Sum naively and you double-count.

In [ ]:
TRACES = []

class BudgetExceeded(Exception): pass
class CircuitBreakerTripped(Exception): pass

BUDGETS = {}           # key -> USD limit
SPEND   = defaultdict(float)

def instrumented_call(prompt, *, model=MODEL, max_tokens=120,
                      business_unit, use_case_id, environment='prod',
                      workload_type='inference', user_id='u-000',
                      session_id=None, cost_tier=2, budget_key=None):
    """One LLM call, fully tagged, budget-enforced, and emitted as an OTel-shaped span."""
    budget_key = budget_key or business_unit

    # --- CONTROL: enforce the budget BEFORE spending, not after ---
    limit = BUDGETS.get(budget_key)
    if limit is not None and SPEND[budget_key] >= limit:
        raise BudgetExceeded(
            f'ExceededBudget: key={budget_key} spend={usd(SPEND[budget_key])} limit={usd(limit)}')

    t0 = time.time()
    r = client.messages.create(model=model, max_tokens=max_tokens,
                               messages=[{'role':'user','content':prompt}])
    u = r.usage
    cr = getattr(u, 'cache_read_input_tokens', 0) or 0
    cw = getattr(u, 'cache_creation_input_tokens', 0) or 0
    c = cost(model, inp=u.input_tokens, out=u.output_tokens, cache_w=cw, cache_r=cr)
    SPEND[budget_key] += c

    span = {
        # --- OTel GenAI semantic conventions (v1.27+) ---
        'gen_ai.provider.name':       'anthropic',
        'gen_ai.request.model':        model,
        'gen_ai.usage.input_tokens':   u.input_tokens,
        'gen_ai.usage.output_tokens':  u.output_tokens,
        'gen_ai.cache.read_tokens':    cr,
        'gen_ai.cache.write_tokens':   cw,
        'gen_ai.client.operation.duration': round(time.time()-t0, 3),
        # --- your attribution tags: the part that makes it a cost model ---
        'business_unit': business_unit,
        'use_case_id':   use_case_id,
        'environment':   environment,
        'workload_type': workload_type,
        'user_id':       user_id,
        'session_id':    session_id or str(uuid.uuid4())[:8],
        'cost_tier':     cost_tier,
        # --- derived ---
        'usd': c,
    }
    TRACES.append(span)
    return r.content[0].text, span

print('instrumented client ready')

---
## 2. Simulate five use cases across three business units

In [ ]:
WORKLOAD = [
  ('support',        'uc-101', 'customer-ops', 'Summarise: shipment NW-1042 delayed 51 hours, customer claim $410.'),
  ('support',        'uc-101', 'customer-ops', 'Summarise: shipment NW-1043 delayed 12 hours, no claim filed.'),
  ('classification', 'uc-102', 'customer-ops', 'Classify sentiment, one word: "Third delay this month. Unacceptable."'),
  ('classification', 'uc-102', 'customer-ops', 'Classify sentiment, one word: "Arrived early, great service."'),
  ('code_review',    'uc-201', 'engineering',  'In one sentence: what is wrong with `for i in range(len(x)): print(x[i])`?'),
  ('code_review',    'uc-201', 'engineering',  'In one sentence: why prefer a context manager over manual file close?'),
  ('rag_answer',     'uc-301', 'sales-mktg',   'One sentence: what is the business case for prompt caching?'),
  ('rag_answer',     'uc-301', 'sales-mktg',   'One sentence: when does self-hosting an LLM beat an API?'),
  ('summarisation',  'uc-302', 'sales-mktg',   'One sentence: summarise the concept of cost per completed task.'),
  ('summarisation',  'uc-302', 'sales-mktg',   'One sentence: summarise why output tokens cost more than input.'),
]

for name, uc, bu, prompt in WORKLOAD:
    _, span = instrumented_call(prompt, business_unit=bu, use_case_id=uc,
                                user_id=f'u-{abs(hash(bu))%900+100}')
    print(f"{bu:<14} {uc:<8} {name:<15} {usd(span['usd']):>11}")
print(f'\n{len(TRACES)} spans captured')

---
## 3. The dashboard — this is Track and Attribute

In [ ]:
df = pd.DataFrame(TRACES)

print('=== SPEND BY BUSINESS UNIT (the chargeback view) ===')
display(df.groupby('business_unit').agg(
    calls=('usd','size'), usd=('usd','sum'),
    in_tok=('gen_ai.usage.input_tokens','sum'),
    out_tok=('gen_ai.usage.output_tokens','sum')).sort_values('usd', ascending=False)
    .style.format({'usd':'${:,.6f}'}))

print('\n=== SPEND BY USE CASE (where to aim optimisation) ===')
top = df.groupby('use_case_id').usd.sum().sort_values(ascending=False)
display(top.to_frame().style.format({'usd':'${:,.6f}'}))
print(f'Top 3 use cases carry {top.head(3).sum()/top.sum():.0%} of spend '
      f'— that is where Modules 1-3 get pointed.')

print('\n=== TAG COVERAGE (target: 95%+ within 30 days) ===')
required = ['business_unit','use_case_id','environment','gen_ai.request.model',
            'workload_type','user_id','session_id','cost_tier']
for tag in required:
    cov = df[tag].notna().mean() if tag in df else 0.0
    print(f'  {tag:<28} {cov:>6.0%}')

---
## 4. CONTROL — the hard cap, and proof that it fires

> LiteLLM's documentation is explicit: *"Every budget is enforced against spend read from
> the database."* A gateway without persistent state cannot enforce a budget at all.
> Governance has an architecture prerequisite.

In [ ]:
BUDGETS['customer-ops'] = SPEND['customer-ops'] + 0.00015   # tiny headroom, so it trips fast
BUDGETS['engineering']  = 5.00                                # plenty of room

print(f"customer-ops budget: {usd(BUDGETS['customer-ops'])}  (current spend {usd(SPEND['customer-ops'])})")
print(f"engineering budget:  {usd(BUDGETS['engineering'])}\n")

blocked = 0
for i in range(8):
    try:
        _, s = instrumented_call(f'Reply with the single word OK. ({i})',
                                 business_unit='customer-ops', use_case_id='uc-101')
        pct = SPEND['customer-ops'] / BUDGETS['customer-ops']
        flag = ' <-- 80% ALERT' if pct >= 0.8 else ''
        print(f"  call {i}: OK  spend={usd(SPEND['customer-ops'])} ({pct:.0%} of budget){flag}")
    except BudgetExceeded as e:
        blocked += 1
        print(f'  call {i}: HTTP 429  {e}')

print(f'\n{blocked} calls blocked by the hard cap.')

### Tenant isolation — one team's overrun must not become everyone's outage

In [ ]:
_, s = instrumented_call('Reply with the single word OK.',
                         business_unit='engineering', use_case_id='uc-201')
print('engineering key still works. Tenant isolation holds.')
print(f"engineering spend: {usd(SPEND['engineering'])} of {usd(BUDGETS['engineering'])}")

---
## 5. Circuit breakers — the control that would have saved Uber's budget

Three **independent** limits, because they catch different failures. A token cap misses a
slow-tool deadlock; a wall-clock cap misses a fast infinite loop that generates cheaply but endlessly.

In [ ]:
BREAKERS = dict(max_steps=8, max_tokens=6000, max_seconds=25, max_usd=0.02)

def run_agent(goal, breakers=BREAKERS):
    """A deliberately looping agent. Watch the breaker terminate it."""
    sid = str(uuid.uuid4())[:8]
    t0, steps, tokens, spend = time.time(), 0, 0, 0.0
    ctx = goal
    while True:
        steps += 1
        # --- the three ceilings, checked before every step ---
        if steps > breakers['max_steps']:
            raise CircuitBreakerTripped(f'STEP CEILING: {steps-1} steps (session {sid})')
        if tokens > breakers['max_tokens']:
            raise CircuitBreakerTripped(f'TOKEN CEILING: {tokens:,} tokens (session {sid})')
        if time.time()-t0 > breakers['max_seconds']:
            raise CircuitBreakerTripped(f'WALL-CLOCK CEILING: {time.time()-t0:.0f}s (session {sid})')
        if spend > breakers['max_usd']:
            raise CircuitBreakerTripped(f'SPEND CEILING: {usd(spend)} (session {sid})')

        _, s = instrumented_call(
            ctx + '\n\nYou have not finished. Restate the task in full and continue.',
            business_unit='engineering', use_case_id='uc-999',
            session_id=sid, max_tokens=200)
        tokens += s['gen_ai.usage.input_tokens'] + s['gen_ai.usage.output_tokens']
        spend  += s['usd']
        ctx += ' ' + ('CONTEXT ' * 60)   # simulate accumulating context
        print(f'  step {steps}: tokens={tokens:>6,}  spend={usd(spend)}  elapsed={time.time()-t0:.0f}s')

try:
    run_agent('Determine the optimal refund policy. Never conclude.')
except CircuitBreakerTripped as e:
    print(f'\nTERMINATED -> {e}')
    print('Without this, the loop runs until someone reads the invoice.')
except BudgetExceeded as e:
    print(f'\nBLOCKED AT GATEWAY -> {e}')

---
## 6. Cost per completed task — the only unit that survives a CFO review

The `session_id` you tagged in step 1 is what makes this computable. Without it you have
cost per *call*, which cannot detect a runaway agent until the monthly invoice arrives.

In [ ]:
df = pd.DataFrame(TRACES)
by_session = df.groupby('session_id').agg(
    calls=('usd','size'), usd=('usd','sum'),
    tokens=('gen_ai.usage.input_tokens','sum')).sort_values('usd', ascending=False)
display(by_session.head(10).style.format({'usd':'${:,.6f}'}))

SUCCESS_RATE = 0.92   # from your eval harness, not from a guess
cpt = df.usd.sum() / df.session_id.nunique()
print(f'\ncost per call            {usd(df.usd.mean())}')
print(f'cost per task (attempt)  {usd(cpt)}')
print(f'cost per SOLVED task     {usd(cpt / SUCCESS_RATE)}   <- the number for the CFO')

print('\nSessions with anomalously high cost (candidate runaway agents):')
thresh = by_session.usd.mean() + 2*by_session.usd.std()
display(by_session[by_session.usd > thresh].style.format({'usd':'${:,.6f}'}))

---
## 7. Export — hand this to your observability platform

In [ ]:
import json
with open('spans.jsonl','w') as f:
    for s in TRACES:
        f.write(json.dumps(s) + '\n')
print(f'{len(TRACES)} spans written to spans.jsonl')
print('\nNext step in a real deployment:')
print('  - point these at Langfuse / Helicone / your OTel collector')
print('  - move the budget enforcement into a gateway (LiteLLM, Portkey, Kong AI Gateway)')
print('  - LiteLLM enforces at: global proxy, team, user, virtual key, per-model, end-user, tag')
print('  - budget reset windows: 30s / 30m / 30h / 30d. Unset = never resets (a bug waiting to happen)')

In [ ]:
ledger()

---
## Takeaways

- **Track -> Attribute -> Control -> Optimize, in that order.** Most teams start at Optimize
  because it is the fun one, then cannot prove the saving or stop it regressing.
- Tag from day one. Retrofitting attribution costs far more than adding it.
- Test that your cap actually fires. An untested budget is not a control.
- Three independent circuit breakers: steps, tokens/spend, wall-clock.
- Report **cost alongside quality** on the same dashboard. Dollars without task success
  rewards exactly the wrong optimisation.